# BottleNeck – Contrôle qualité automatisé des données

**Projet amélioré du projet 6 · Paul Chipponi · octobre 2026**

## Contexte

BottleNeck, caviste en ligne, pilote son activité à partir de trois exports mensuels : l'ERP (prix, stock, prix d'achat), l'export WordPress du site (ventes, descriptions) et une table de liaison maintenue à la main. Le projet 6 vérifiait la qualité de ces données à la main, une seule fois, et supprimait les lignes en anomalie avant de calculer les indicateurs présentés au CODIR.

Ce notebook transforme ces vérifications en règles de validation automatiques et réutilisables, écrites avec Pandera. Il applique un principe simple : **signaler sans supprimer**. Une anomalie est détectée, décrite et rapportée à l'équipe métier ; elle n'écarte une ligne d'un calcul que lorsqu'elle rend ce calcul faux.

## Sommaire

1. Environnement et chargement des données
2. Règles de validation
3. Fonction de contrôle unique
4. Test de robustesse sur un export dégradé
5. Rapport d'anomalies pour l'équipe métier
6. Recalcul des indicateurs du projet 6
7. Conclusions et limites

Pour exécuter le notebook : placer les fichiers `erp.xlsx`, `web.xlsx` et `liaison.xlsx` dans le même dossier, installer les dépendances (`pip install -r requirements.txt`), puis lancer *Kernel > Restart Kernel and Run All Cells*.

## 1. Environnement et chargement des données

Les versions des bibliothèques sont affichées pour garantir la reproductibilité ; elles sont figées dans `requirements.txt`. Les avertissements d'openpyxl au chargement des fichiers Excel sont sans conséquence et sont masqués.

In [1]:
import re
import sys
import warnings
from importlib.metadata import version

import numpy as np
import pandas as pd
import pandera.pandas as pa
from openpyxl.styles import Font, Alignment

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

print("Python :", sys.version.split()[0])
for paquet in ["pandas", "numpy", "pandera", "openpyxl"]:
    print(f"{paquet} : {version(paquet)}")

Python : 3.13.11
pandas : 3.0.2
numpy : 2.4.4
pandera : 0.34.1
openpyxl : 3.1.5


In [2]:
erp = pd.read_excel("erp.xlsx")
web = pd.read_excel("web.xlsx")
liaison = pd.read_excel("liaison.xlsx")

print("ERP :", erp.shape, "| Web :", web.shape, "| Liaison :", liaison.shape)

ERP : (825, 6) | Web : (1513, 29) | Liaison : (825, 2)


## 2. Règles de validation

Les 15 règles du catalogue métier (tâche 2) sont traduites en contrôles. Chaque contrôle porte un message en français préfixé par l'identifiant de sa règle, pour que le rapport soit lisible par l'équipe de Nicolas. Une règle est **bloquante** lorsque sa violation fausse un indicateur présenté au CODIR, et relève de l'**avertissement** sinon. Les erreurs de structure (type de colonne, valeur hors liste autorisée) sont regroupées sous la mention *Structure*.

In [3]:
GRAVITE = {
    "R1": "Bloquante", "R2": "Bloquante", "R3": "Avertissement", "R4": "Bloquante",
    "R5": "Bloquante", "R6": "Avertissement", "R7": "Avertissement", "R8": "Bloquante",
    "R9": "Bloquante", "R10": "Avertissement", "R11": "Avertissement", "R12": "Avertissement",
    "R13": "Bloquante", "R14": "Avertissement", "R15": "Avertissement", "Structure": "Bloquante",
}

### 2.1 ERP – règles R1 à R4

R1 et R2 portent sur une seule colonne. R3 et R4 comparent deux colonnes d'une même ligne et sont donc définies au niveau du tableau. R4 exclut les prix négatifs ou nuls, déjà signalés par R1, pour qu'une même erreur ne soit pas comptée deux fois.

In [4]:
schema_erp = pa.DataFrameSchema(
    columns={
        "product_id": pa.Column(int, unique=True, nullable=False),
        "onsale_web": pa.Column(int, pa.Check.isin([0, 1]), nullable=False),
        "price": pa.Column(float, pa.Check.gt(0, name="R1",
                  error="R1 – Le prix de vente doit être strictement positif"), nullable=False),
        "stock_quantity": pa.Column(int, pa.Check.ge(0, name="R2",
                  error="R2 – La quantité en stock ne doit jamais être négative"), nullable=False),
        "stock_status": pa.Column(checks=pa.Check.isin(["instock", "outofstock"]), nullable=False),
        "purchase_price": pa.Column(float, nullable=False),
    },
    checks=[
        # R3 : en stock si quantité > 0, en rupture sinon
        pa.Check(lambda df: ((df["stock_quantity"] > 0) & (df["stock_status"] == "instock"))
                          | ((df["stock_quantity"] <= 0) & (df["stock_status"] == "outofstock")),
                 name="R3", error="R3 – Le statut de stock ne correspond pas à la quantité"),
        # R4 : prix de vente >= prix d'achat (les prix négatifs relèvent déjà de R1)
        pa.Check(lambda df: (df["price"] <= 0) | (df["price"] >= df["purchase_price"]),
                 name="R4", error="R4 – Le prix de vente est inférieur au prix d'achat"),
    ],
)

### 2.2 Table de liaison – règle R5

L'identifiant web peut être vide : les produits vendus uniquement en magasin n'en ont pas. Pandera ignore automatiquement les valeurs vides dans ce contrôle.

In [5]:
schema_liaison = pa.DataFrameSchema(
    columns={
        "product_id": pa.Column(int, unique=True, nullable=False),
        "id_web": pa.Column(
            checks=pa.Check(lambda s: s.astype(str).str.fullmatch(r"\d+"), name="R5",
                            error="R5 – L'identifiant web doit être numérique"),
            nullable=True,
        ),
    },
)

### 2.3 Export web – règles R5, R8, R9, R11 et R14

L'export du site mélange des produits et d'autres lignes (images, lignes vides). R8 signale ces lignes parasites ; toutes les autres règles ne s'appliquent qu'aux vrais produits. La détection du code HTML (R11) couvre les balises (`<div>`) et les entités (`&amp;`, `&nbsp;`), que le nettoyage du projet 6 laissait passer.

In [6]:
MOTIF_HTML = r"<[^>]+>|&[a-zA-Z]+;|&#\d+;"


def est_produit(df):
    return df["post_type"] == "product"


def sku_en_doublon(df):
    produits = est_produit(df) & df["sku"].notna()
    doublons = df.loc[produits, "sku"].astype(str).duplicated(keep=False)
    return doublons.reindex(df.index, fill_value=False)


schema_web = pa.DataFrameSchema(
    checks=[
        pa.Check(lambda df: est_produit(df), name="R8",
                 error="R8 – La ligne n'est pas un produit (image, ligne vide…)"),
        pa.Check(lambda df: ~sku_en_doublon(df), name="R8",
                 error="R8 – Le produit apparaît plusieurs fois dans l'export"),
        pa.Check(lambda df: ~est_produit(df) | df["sku"].isna()
                            | df["sku"].astype(str).str.fullmatch(r"\d+"), name="R5",
                 error="R5 – La référence web doit être numérique"),
        pa.Check(lambda df: ~est_produit(df) | (df["sku"].notna() & (df["total_sales"] >= 0)), name="R9",
                 error="R9 – Le produit doit avoir une référence et des ventes positives ou nulles"),
        pa.Check(lambda df: ~est_produit(df)
                            | ~(df["post_title"].astype(str).str.contains(MOTIF_HTML)
                                | df["post_excerpt"].astype(str).str.contains(MOTIF_HTML)), name="R11",
                 error="R11 – Le titre ou la description contient du code HTML"),
        pa.Check(lambda df: ~est_produit(df) | df["product_type"].notna(), name="R14",
                 error="R14 – Le produit n'a pas de type (ex. bon cadeau)"),
    ],
)

### 2.4 Règles R10, inter-tables et R15

Ces règles portent sur des colonnes (R10) ou comparent plusieurs tables (R6, R7, R12, R13, R15) : elles sont exprimées en pandas, plus lisible pour ce type de contrôle, et intégrées directement dans la fonction de contrôle ci-dessous.

R10 distingue deux cas : une colonne **vide** est à retirer de l'export, une colonne **constante** est à confirmer avec Nicolas, car une valeur unique aujourd'hui peut porter une information demain (statut de publication, statut fiscal).

Les identifiants web mélangent nombres et texte (règle R5) : pour toutes les comparaisons entre tables, ils sont convertis en texte.

## 3. Fonction de contrôle unique

La fonction `controler` applique l'ensemble des règles à trois tables et renvoie un tableau unique d'anomalies, **sans modifier les données**. Elle produit une seule ligne par élément en anomalie (Pandera renvoie sinon une ligne par colonne pour les règles qui en comparent plusieurs), et les erreurs sans identifiant de règle sont signalées sous *Structure* au lieu d'être ignorées.

In [7]:
def lignes_pandera(schema, df, table, colonne_id):
    """Valide un schéma Pandera et renvoie une ligne par anomalie et par élément concerné."""
    try:
        schema.validate(df, lazy=True)
        return []
    except pa.errors.SchemaErrors as e:
        cas = e.failure_cases
    # Une seule ligne par couple (contrôle, ligne du tableau), quel que soit le nombre de colonnes
    cas = cas.drop_duplicates(subset=["check", "index"])
    lignes = []
    for _, c in cas.iterrows():
        message = str(c["check"])
        trouve = re.match(r"(R\d+)", message)
        regle = trouve.group(1) if trouve else "Structure"
        if regle == "Structure":
            message = f"Structure – {c['column']} : contrôle « {message} » non respecté ({c['failure_case']})"
        if pd.notna(c["index"]):
            ident = df.loc[c["index"], colonne_id]
            ident = c["index"] if pd.isna(ident) else ident
        else:
            ident = c["column"]
        lignes.append({"Règle": regle, "Table": table, "Identifiant": str(ident), "Message": message})
    return lignes


def controler(erp, web, liaison):
    """Applique les 15 règles et renvoie un tableau unique d'anomalies, sans modifier les données."""
    lignes = []
    lignes += lignes_pandera(schema_erp, erp, "ERP", "product_id")
    lignes += lignes_pandera(schema_liaison, liaison, "Liaison", "product_id")
    lignes += lignes_pandera(schema_web, web, "Web", "sku")

    # R10 : colonnes vides ou constantes
    nb = web.nunique(dropna=True)
    for col in nb[nb == 0].index:
        lignes.append({"Règle": "R10", "Table": "Web", "Identifiant": col,
                       "Message": "R10 – Colonne vide : à retirer de l'export"})
    for col in nb[nb == 1].index:
        lignes.append({"Règle": "R10", "Table": "Web", "Identifiant": col,
                       "Message": "R10 – Colonne constante : utilité à confirmer avec Nicolas"})

    # Règles inter-tables, identifiants comparés en texte
    liaison_txt = liaison.assign(id_web=liaison["id_web"].astype("string"))
    produits_web = web[web["post_type"] == "product"].dropna(subset=["sku"])
    produits_web = produits_web.assign(sku=produits_web["sku"].astype(str))
    skus_web = set(produits_web["sku"])
    ids_liaison = set(liaison_txt["id_web"].dropna())
    erp_liaison = erp.merge(liaison_txt, on="product_id", how="left")
    erp_titres = erp_liaison.merge(produits_web[["sku", "post_title"]],
                                   left_on="id_web", right_on="sku", how="left")

    controles = [
        ("R6", "ERP / liaison", erp_liaison[(erp_liaison["onsale_web"] == 1) & erp_liaison["id_web"].isna()],
         "product_id", "R6 – Produit en vente web sans identifiant web"),
        ("R7", "ERP / liaison", erp_liaison[(erp_liaison["onsale_web"] == 0) & erp_liaison["id_web"].notna()],
         "product_id", "R7 – Produit retiré de la vente web avec un identifiant web"),
        ("R12", "Liaison / web", liaison_txt[liaison_txt["id_web"].notna() & ~liaison_txt["id_web"].isin(skus_web)],
         "product_id", "R12 – Référence de la liaison absente du site"),
        ("R13", "Web / liaison", produits_web[~produits_web["sku"].isin(ids_liaison)],
         "sku", "R13 – Produit du site absent de la liaison et de l'ERP"),
        ("R15", "ERP", erp_titres[erp_titres["post_title"].isna()],
         "product_id", "R15 – Aucun libellé disponible, ni dans l'ERP ni sur le site"),
    ]
    for regle, table, df, col, message in controles:
        for ident in df[col]:
            lignes.append({"Règle": regle, "Table": table, "Identifiant": str(ident), "Message": message})

    rapport = pd.DataFrame(lignes)
    rapport.insert(1, "Gravité", rapport["Règle"].map(GRAVITE))
    return rapport

In [8]:
octobre = controler(erp, web, liaison)

print("Anomalies détectées sur l'export d'octobre :", len(octobre))
display(octobre.groupby(["Règle", "Gravité"]).size().rename("Anomalies").reset_index())

Anomalies détectées sur l'export d'octobre : 1170


,Règle,Gravité,Anomalies
0,R1,Bloquante,3
1,R10,Avertissement,16
2,R11,Avertissement,183
3,R12,Avertissement,20
4,R14,Avertissement,1
5,R15,Avertissement,111
6,R2,Bloquante,2
7,R3,Avertissement,2
8,R4,Bloquante,4
9,R5,Bloquante,5


## 4. Test de robustesse sur un export dégradé

Un seul mois de données étant disponible, les règles ont été testées sur un export volontairement dégradé, produit par le notebook `export_degrade.ipynb` : 13 erreurs connues ont été injectées dans des copies des fichiers, sur des produits sans anomalie tirés au hasard avec une graine fixe (`random_state=42`). Le test vérifie que chaque erreur injectée est détectée par sa règle. Les autres anomalies nouvelles sont des effets de cascade d'une même injection (par exemple, un stock négatif rend aussi le statut incohérent).

In [9]:
import os

fichiers_degrades = ["erp_degrade.xlsx", "web_degrade.xlsx", "liaison_degrade.xlsx", "injections.csv"]

if all(os.path.exists(f) for f in fichiers_degrades):
    degrade = controler(pd.read_excel("erp_degrade.xlsx"),
                        pd.read_excel("web_degrade.xlsx"),
                        pd.read_excel("liaison_degrade.xlsx"))
    cle = ["Règle", "Identifiant", "Message"]
    nouvelles = (degrade.merge(octobre[cle], on=cle, how="left", indicator=True)
                        .query("_merge == 'left_only'").drop(columns="_merge"))

    injections = pd.read_csv("injections.csv", dtype=str)
    injections["Détectée"] = [
        ((nouvelles["Règle"] == r["Règle"]) & (nouvelles["Identifiant"] == r["Identifiant"])).any()
        for _, r in injections.iterrows()
    ]
    print("Anomalies octobre :", len(octobre), "| export dégradé :", len(degrade),
          "| nouvelles :", len(nouvelles))
    display(injections)
    print(f"Taux de détection : {injections['Détectée'].mean():.0%}")
else:
    print("Fichiers dégradés absents : exécuter d'abord le notebook export_degrade.ipynb.")

Anomalies octobre : 1170 | export dégradé : 1192 | nouvelles : 22


,Règle,Table,Identifiant,Modification,Détectée
0,R1,ERP,6221,Prix de vente mis à 0,True
1,R2,ERP,4275,Stock mis à -5,True
2,R3,ERP,6585,"Stock mis à 0, statut instock conservé",True
3,R4,ERP,4632,Prix de vente égal à la moitié du prix d'achat,True
4,R5,Liaison,5764,Lettre ajoutée à l'identifiant web,True
5,R6,Liaison,4920,Identifiant web supprimé pour un produit en ve...,True
6,R8,Web,14509,Ligne produit dupliquée,True
7,R9,Web,15481,Ventes mises à -3,True
8,R10,Web,colonne_test_vide,Colonne entièrement vide ajoutée,True
9,R11,Web,15856,Balise HTML ajoutée au titre,True


Taux de détection : 100%


## 5. Rapport d'anomalies pour l'équipe métier

Le rapport est un fichier Excel, `rapport_anomalies.xlsx`, choisi parce que l'équipe de BottleNeck l'utilise déjà, peut le trier et le filtrer sans compétence technique, et parce qu'il est compatible avec les lecteurs d'écran.

- L'onglet **Synthèse** présente une ligne par règle (énoncé métier, gravité, nombre d'anomalies, outil où corriger, responsable pressenti), y compris les règles sans anomalie, pour prouver qu'elles ont été exécutées.
- L'onglet **Détail** présente une ligne par anomalie, avec le nom du produit lorsqu'il est connu. Les lignes d'images de l'export (R8) y sont regroupées en une seule ligne, puisqu'une seule correction suffit.

In [10]:
CATALOGUE = pd.DataFrame([
    ("R1", "Le prix de vente est strictement positif", "ERP", "Équipe ventes"),
    ("R2", "La quantité en stock n'est jamais négative", "ERP", "Équipe ventes"),
    ("R3", "Le statut de stock correspond à la quantité", "ERP", "Équipe ventes"),
    ("R4", "Le prix de vente est supérieur ou égal au prix d'achat, sauf déstockage validé", "ERP", "Équipe ventes, validation de Nicolas"),
    ("R5", "L'identifiant web est toujours numérique", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R6", "Un produit en vente sur le site a un identifiant web", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R7", "Un produit retiré de la vente web n'a plus d'identifiant web actif", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R8", "L'export web ne contient que des produits, chacun une seule fois", "Paramétrage de l'export du site", "Responsable du site e-commerce"),
    ("R9", "Chaque produit du site a une référence et des ventes positives ou nulles", "Site (WordPress)", "Responsable du site e-commerce"),
    ("R10", "L'export ne contient pas de colonnes vides ou inutilisées", "Paramétrage de l'export du site", "Responsable du site e-commerce"),
    ("R11", "Les titres et descriptions ne contiennent pas de code HTML", "Site (WordPress)", "Responsable du site e-commerce"),
    ("R12", "Toute référence de la table de liaison existe sur le site", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R13", "Tout produit vendu sur le site existe dans l'ERP", "ERP et table de liaison", "Équipe ventes et stagiaire"),
    ("R14", "Les produits qui ne sont pas des vins ont un type dédié", "Site (WordPress)", "Responsable du site e-commerce"),
    ("R15", "Chaque produit de l'ERP a un libellé disponible, dans l'ERP ou sur le site", "ERP", "Équipe ventes"),
    ("Structure", "Les valeurs respectent le format attendu (types, valeurs autorisées)", "Outil d'origine de la donnée", "À déterminer selon la table"),
], columns=["Règle", "Énoncé métier", "Où corriger", "Responsable pressenti"])

MESSAGE_R8_LIGNES = "R8 – La ligne n'est pas un produit (image, ligne vide…)"


def nom_produit(rapport, erp, web, liaison):
    """Ajoute le nom du produit à chaque anomalie, quand il est connu."""
    produits = web[web["post_type"] == "product"].dropna(subset=["sku"])
    nom_par_sku = dict(zip(produits["sku"].astype(str), produits["post_title"]))
    sku_par_pid = dict(zip(liaison["product_id"].astype(str), liaison["id_web"].astype("string")))

    def trouver(ident):
        if ident in nom_par_sku:
            return nom_par_sku[ident]
        sku = sku_par_pid.get(ident)
        return nom_par_sku.get(sku, "") if pd.notna(sku) else ""

    return rapport.assign(Produit=rapport["Identifiant"].map(trouver))


def regrouper_r8(rapport):
    """Remplace les lignes non-produits de R8 par une seule ligne récapitulative."""
    lignes_r8 = rapport["Message"] == MESSAGE_R8_LIGNES
    nb = int(lignes_r8.sum())
    if nb == 0:
        return rapport
    recap = pd.DataFrame([{"Règle": "R8", "Gravité": GRAVITE["R8"], "Table": "Web",
                           "Identifiant": f"{nb} lignes",
                           "Message": f"R8 – {nb} lignes d'images ou lignes vides dans l'export : "
                                      "une seule correction, dans le paramétrage de l'export"}])
    return pd.concat([rapport[~lignes_r8], recap], ignore_index=True)


def en_nombre(valeur):
    """Convertit un identifiant en nombre entier quand c'est possible, pour un tri correct dans Excel."""
    return int(valeur) if str(valeur).isdigit() else valeur


def exporter_rapport(rapport, erp, web, liaison, fichier="rapport_anomalies.xlsx"):
    """Produit le rapport Excel : un onglet Synthèse et un onglet Détail."""
    ordre = {"Bloquante": 0, "Avertissement": 1}

    detail = (nom_produit(regrouper_r8(rapport), erp, web, liaison)
              .merge(CATALOGUE[["Règle", "Où corriger"]], on="Règle", how="left")
              .assign(_o=lambda d: d["Gravité"].map(ordre))
              .sort_values(["_o", "Règle", "Identifiant"]).drop(columns="_o")
              [["Gravité", "Règle", "Table", "Identifiant", "Produit", "Message", "Où corriger"]])
    detail["Identifiant"] = detail["Identifiant"].map(en_nombre).astype(object)

    comptes = rapport.groupby("Règle").size().rename("Nombre d'anomalies")
    synthese = (CATALOGUE.assign(Gravité=CATALOGUE["Règle"].map(GRAVITE))
                .merge(comptes, left_on="Règle", right_index=True, how="left")
                .fillna({"Nombre d'anomalies": 0})
                .astype({"Nombre d'anomalies": int})
                .assign(_o=lambda d: d["Gravité"].map(ordre),
                        _n=lambda d: d["Règle"].str.extract(r"(\d+)")[0].astype(float))
                .sort_values(["_o", "_n"]).drop(columns=["_o", "_n"])
                [["Gravité", "Règle", "Énoncé métier", "Nombre d'anomalies", "Où corriger", "Responsable pressenti"]])

    with pd.ExcelWriter(fichier, engine="openpyxl") as writer:
        synthese.to_excel(writer, sheet_name="Synthèse", index=False)
        detail.to_excel(writer, sheet_name="Détail", index=False)
        for nom, df in [("Synthèse", synthese), ("Détail", detail)]:
            feuille = writer.sheets[nom]
            feuille.freeze_panes = "A2"                    # en-tête toujours visible
            feuille.auto_filter.ref = feuille.dimensions   # filtres sur chaque colonne
            for cellule in feuille[1]:
                cellule.font = Font(bold=True)
            for i, col in enumerate(df.columns, start=1):
                largeur = min(max(len(str(col)), df[col].astype(str).str.len().max()) + 2, 70)
                feuille.column_dimensions[feuille.cell(1, i).column_letter].width = largeur
                for colonne in feuille.iter_cols(min_col=i, max_col=i, min_row=2):
                    for c in colonne:
                        c.alignment = Alignment(wrap_text=True, vertical="top")
    return synthese, detail

In [11]:
synthese, detail = exporter_rapport(octobre, erp, web, liaison)

print("Rapport enregistré : rapport_anomalies.xlsx")
print("Lignes dans l'onglet Détail :", len(detail))
display(synthese)

Rapport enregistré : rapport_anomalies.xlsx
Lignes dans l'onglet Détail : 374


,Gravité,Règle,Énoncé métier,Nombre d'anomalies,Où corriger,Responsable pressenti
0,Bloquante,R1,Le prix de vente est strictement positif,3,ERP,Équipe ventes
1,Bloquante,R2,La quantité en stock n'est jamais négative,2,ERP,Équipe ventes
3,Bloquante,R4,Le prix de vente est supérieur ou égal au prix...,4,ERP,"Équipe ventes, validation de Nicolas"
4,Bloquante,R5,L'identifiant web est toujours numérique,5,Table de liaison,Stagiaire en charge de la liaison
7,Bloquante,R8,"L'export web ne contient que des produits, cha...",797,Paramétrage de l'export du site,Responsable du site e-commerce
8,Bloquante,R9,Chaque produit du site a une référence et des ...,2,Site (WordPress),Responsable du site e-commerce
12,Bloquante,R13,Tout produit vendu sur le site existe dans l'ERP,0,ERP et table de liaison,Équipe ventes et stagiaire
15,Bloquante,Structure,Les valeurs respectent le format attendu (type...,0,Outil d'origine de la donnée,À déterminer selon la table
2,Avertissement,R3,Le statut de stock correspond à la quantité,2,ERP,Équipe ventes
5,Avertissement,R6,Un produit en vente sur le site a un identifia...,3,Table de liaison,Stagiaire en charge de la liaison


## 6. Recalcul des indicateurs du projet 6

Les indicateurs présentés au CODIR dans le projet 6 sont recalculés selon le principe *signaler sans supprimer*, puis comparés aux valeurs d'origine. Les règles de traitement sont les suivantes :

| Indicateur | Traitement |
|---|---|
| Chiffre d'affaires web | Tous les produits du site reliés à l'ERP, y compris ceux qui ont une erreur de stock |
| Lignes non-produits (R8) | Exclues des calculs, signalées dans le rapport |
| Pareto 20/80 | Calculé sur les seuls produits vendus sur le site |
| Valeur du stock | Stock négatif compté à zéro, signalé |
| Marges | Produits à prix valide (R1), avec un second calcul hors ventes à perte (R4) |

In [12]:
# Valeurs présentées dans le projet 6
CA_P6 = 143324.10
PARETO_P6 = 44.1
STOCK_P6 = 298540.07
MARGE_LATENTE_P6 = 233406.13

### 6.1 Chiffre d'affaires web

In [13]:
produits_site = web[web["post_type"] == "product"].dropna(subset=["sku"])
produits_site = produits_site.assign(sku=produits_site["sku"].astype(str))
liaison_txt = liaison.assign(id_web=liaison["id_web"].astype("string"))

ventes = (produits_site[["sku", "post_title", "total_sales"]]
          .merge(liaison_txt, left_on="sku", right_on="id_web", how="inner")
          .merge(erp, on="product_id", how="inner"))
ventes["CA"] = ventes["price"] * ventes["total_sales"]

ca_total = ventes["CA"].sum()
print("Produits vendus sur le site et reliés à l'ERP :", len(ventes))
print(f"CA web recalculé : {ca_total:,.2f} € (P6 : {CA_P6:,.2f} €, écart : {ca_total - CA_P6:,.2f} €)")

# Produits pris en compte ici mais supprimés par le nettoyage du projet 6 (prix ou stock négatif)
supprimes_p6 = erp.loc[(erp["price"] < 0) | (erp["stock_quantity"] < 0), "product_id"]
display(ventes[ventes["product_id"].isin(supprimes_p6)][["product_id", "post_title", "price", "total_sales", "CA"]])

Produits vendus sur le site et reliés à l'ERP : 714
CA web recalculé : 143,680.10 € (P6 : 143,324.10 €, écart : 356.00 €)


,product_id,post_title,price,total_sales,CA
536,5700,"Gilles Robin Crozes-Hermitage Rouge ""1920"" 2016",44.5,8.0,356.0


L'écart correspond au produit 5700, dont le stock est saisi à -1. Le projet 6 avait supprimé cette ligne avant la jointure : ses 8 ventes avaient disparu du chiffre d'affaires, et sa ligne web orpheline avait été présentée comme un « produit fantôme ».

### 6.2 Concentration du chiffre d'affaires (Pareto)

In [14]:
ca_tries = ventes["CA"].sort_values(ascending=False).reset_index(drop=True)
part_cumulee = ca_tries.cumsum() / ca_tries.sum()

nb_top20 = int(np.ceil(0.20 * len(ca_tries)))
part_top20 = ca_tries.head(nb_top20).sum() / ca_tries.sum() * 100

nb_pour_80 = int((part_cumulee < 0.80).sum()) + 1
part_produits_80 = nb_pour_80 / len(ca_tries) * 100

print(f"Produits du site : {len(ca_tries)} (dont {int((ventes['CA'] == 0).sum())} sans vente en octobre)")
print(f"Les 20 % meilleurs produits ({nb_top20}) réalisent {part_top20:.1f} % du CA (P6 : {PARETO_P6} %)")
print(f"Il faut {nb_pour_80} produits, soit {part_produits_80:.1f} % du catalogue web, pour atteindre 80 % du CA")

Produits du site : 714 (dont 25 sans vente en octobre)
Les 20 % meilleurs produits (143) réalisent 40.2 % du CA (P6 : 44.1 %)
Il faut 435 produits, soit 60.9 % du catalogue web, pour atteindre 80 % du CA


Le projet 6 calculait ce ratio sur 820 références, dont une centaine vendues uniquement en magasin et sans chiffre d'affaires web, ce qui gonflait la part des « 20 % ».

### 6.3 Valeur du stock et marges

In [15]:
e = erp.copy()
e["stock_valide"] = e["stock_quantity"].clip(lower=0)            # stock négatif compté à 0
prix_valide = e["price"] > 0                                      # R1 respectée
vente_a_perte = prix_valide & (e["price"] < e["purchase_price"])  # R4

valeur_stock = (e["stock_valide"] * e["purchase_price"]).sum()
marge_latente = e["stock_valide"] * (e["price"] - e["purchase_price"])
ml_toutes = marge_latente[prix_valide].sum()
ml_sans_r4 = marge_latente[prix_valide & ~vente_a_perte].sum()

print(f"Valeur du stock : {valeur_stock:,.2f} € (P6 : {STOCK_P6:,.2f} €, écart : {valeur_stock - STOCK_P6:,.2f} €)")
print(f"Marge latente, tous produits à prix valide : {ml_toutes:,.2f} € (P6 : {MARGE_LATENTE_P6:,.2f} €)")
print(f"Marge latente, hors ventes à perte (R4)   : {ml_sans_r4:,.2f} €")
print(f"Effet des ventes à perte sur la marge     : {ml_toutes - ml_sans_r4:,.2f} €")

display(e[vente_a_perte][["product_id", "price", "purchase_price", "stock_valide"]]
        .assign(marge_latente=marge_latente[vente_a_perte]))

Valeur du stock : 298,627.66 € (P6 : 298,540.07 €, écart : 87.59 €)
Marge latente, tous produits à prix valide : 233,406.13 € (P6 : 233,406.13 €)
Marge latente, hors ventes à perte (R4)   : 239,831.64 €
Effet des ventes à perte sur la marge     : -6,425.51 €


,product_id,price,purchase_price,stock_valide,marge_latente
210,4355,12.65,77.48,97,-6288.51
391,4864,8.30,9.99,0,-0.00
724,6324,92.00,99.00,18,-126.00
817,7196,31.00,31.20,55,-11.00


### 6.4 Synthèse des écarts avec le projet 6

In [16]:
synthese_indicateurs = pd.DataFrame([
    ("Chiffre d'affaires web (€)", CA_P6, ca_total, "Produit 5700 supprimé par le nettoyage du projet 6"),
    ("Part du CA des 20 % meilleurs produits (%)", PARETO_P6, part_top20, "Dénominateur incluant les produits vendus en magasin"),
    ("Valeur du stock au prix d'achat (€)", STOCK_P6, valeur_stock, "Produit 6594 supprimé par le nettoyage du projet 6"),
    ("Marge latente (€)", MARGE_LATENTE_P6, ml_toutes, "Identique, mais intègre les ventes à perte non repérées"),
], columns=["Indicateur", "Projet 6", "Recalculé", "Cause de l'écart"])
synthese_indicateurs["Écart"] = synthese_indicateurs["Recalculé"] - synthese_indicateurs["Projet 6"]
display(synthese_indicateurs.round(2))

,Indicateur,Projet 6,Recalculé,Cause de l'écart,Écart
0,Chiffre d'affaires web (€),143324.10,143680.10,Produit 5700 supprimé par le nettoyage du proj...,356.00
1,Part du CA des 20 % meilleurs produits (%),44.10,40.24,Dénominateur incluant les produits vendus en m...,-3.86
2,Valeur du stock au prix d'achat (€),298540.07,298627.66,Produit 6594 supprimé par le nettoyage du proj...,87.59
3,Marge latente (€),233406.13,233406.13,"Identique, mais intègre les ventes à perte non...",0.00


## 7. Conclusions et limites

**Conclusions.** Les 15 règles du catalogue sont automatisées et détectent 100 % des erreurs injectées dans l'export dégradé, sans fausse alerte. Le principe *signaler sans supprimer* corrige trois résultats du projet 6 : le chiffre d'affaires web (+356 €, produit 5700), la concentration du chiffre d'affaires (40,2 % au lieu de 44,1 %) et l'absence supposée d'erreur de prix (quatre ventes à perte, dont le produit 4355 qui pèse à lui seul 6 288 € de marge latente). Le « produit fantôme » du projet 6 était un effet de bord de son propre nettoyage.

**Limites.**

- Les règles n'ont été testées que sur un mois de données réelles et sur un export dégradé construit à partir du catalogue de règles.
- La règle R4 ne distingue pas une vente à perte volontaire d'une erreur de saisie sans la liste des déstockages validés par Nicolas.
- Les règles R10, inter-tables et R15 sont exprimées en pandas et non en Pandera, ce qui répartit les contrôles entre deux mécanismes.
- Pour les lignes du site sans référence (R9), l'identifiant affiché dans le rapport est le numéro de ligne de l'export.
- Le rapport ne distingue pas encore les anomalies d'origine de leurs effets de cascade.